# 过度识别检验（工具变量回归）

> 对应代码：`MathStatsCode/code_in_notes/Chap.13/overid_reg_sq.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用 x、x²、x³ 作为工具变量进行 GMM 估计，并用 estat overid 检验过度识别约束。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.13`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.13")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear`、`set seed 250211`、`set obs 100`：准备100个观测的模拟环境。


In [ ]:
%%stata
clear
set seed 250211
set obs 100


**DGP**

生成回归数据：`x~N(0,4)`；由 x 派生工具变量 `x2=x²`、`x3=x³`（它们与 x 相关而与误差独立）；`y=1+2x+x²+2ε`——注意因变量中还含有 x² 项，用来观察“用线性模型拟合非线性数据”时工具变量矩条件是否真的成立。


In [ ]:
%%stata
gen x  = rnormal()*2
gen x2 = x^2
gen x3 = x^3
gen y  = 1+2*x+x2+rnormal()*2


**GMM est and test**

`gmm (y-{xb:x}-{alpha}), instruments(x x2 x3)`：以 x、x²、x³ 三个工具变量建立三个矩条件（E[工具×(y-α-βx)]=0），而参数只有两个（α、β）——**过度识别**（3个矩条件、2个参数）。`estat overid` 执行过度识别检验：若数据生成过程确实满足线性模型（矩条件正确），检验不拒绝；反之则拒绝。对比 `ivregress 2sls`，`gmm` 允许灵活设定矩条件并直接提供该检验。


In [ ]:
%%stata
gmm (y-{xb: x}-{alpha}), instruments(x x2 x3)
estat overid
